# Formateo de datos 

In [10]:
import os, duckdb

db_path = "../../db/olist_analytics.duckdb"
views_dir = "../../data/views/"
os.makedirs(views_dir, exist_ok=True)

con = duckdb.connect(db_path, read_only=False)
con.execute("CREATE SCHEMA IF NOT EXISTS olist_fmt")

In [11]:
# SALES (orden)
con.execute("""
CREATE OR REPLACE TABLE olist_fmt.sales AS
SELECT
  CAST(order_id           AS VARCHAR)       AS order_id,
  CAST(items_per_order    AS INTEGER)       AS items_per_order,
  CAST(price              AS DECIMAL(18,2)) AS price,
  CAST(freight_value      AS DECIMAL(18,2)) AS freight_value,
  CAST(order_total_value  AS DECIMAL(18,2)) AS order_total_value,
  CAST(seller_count       AS INTEGER)       AS seller_count,
  CAST(order_year         AS INTEGER)       AS order_year,
  CAST(order_month        AS VARCHAR)       AS order_month,
  CAST(order_dow          AS INTEGER)       AS order_dow
FROM olist.vw_sales;
""")

In [12]:
# LOGISTICS (orden)
con.execute("""
CREATE OR REPLACE TABLE olist_fmt.logistics AS
SELECT
  CAST(v.order_id              AS VARCHAR)        AS order_id,
  CAST(v.delivery_days         AS DECIMAL(10,2))  AS delivery_days,
  CAST(v.estimated_days        AS DECIMAL(10,2))  AS estimated_days,
  CAST(v.delay_vs_estimated    AS DECIMAL(10,2))  AS delay_vs_estimated,
  CAST(v.late_days             AS DECIMAL(10,2))  AS late_days,
  CAST(v.on_time               AS BOOLEAN)        AS on_time,
  CAST(v.prep_hours            AS DECIMAL(10,2))  AS prep_hours,
  CAST(v.transit_days          AS DECIMAL(10,2))  AS transit_days,
  CAST(v.order_year            AS INTEGER)        AS order_year,
  CAST(v.order_month           AS VARCHAR)        AS order_month,
  CAST(v.order_week            AS INTEGER)        AS order_week,
  CAST(v.order_dow             AS INTEGER)        AS order_dow,
  CAST(v.purchase_hour         AS INTEGER)        AS purchase_hour,
  CAST(v.is_weekend_purchase   AS BOOLEAN)        AS is_weekend_purchase,
  CAST(v.delay_bucket          AS VARCHAR)        AS delay_bucket,
  CAST(v.delivery_days_bucket  AS VARCHAR)        AS delivery_days_bucket
FROM olist.vw_logistics AS v;
""")

In [13]:
# CUSTOMER SATISFACTION (orden)
con.execute("""
CREATE OR REPLACE TABLE olist_fmt.customer_satisfaction AS
SELECT
  CAST(order_id AS VARCHAR) AS order_id,
  CAST(review_score AS INTEGER) AS review_score,
  CAST(review_creation_date AS TIMESTAMP) AS review_creation_date,
  CAST(review_answer_timestamp AS TIMESTAMP) AS review_answer_timestamp,
  CAST(review_after_delivery_hours AS INTEGER) AS review_after_delivery_hours
FROM olist.vw_customer_satisfaction;
""")

In [14]:
# SELLERS (vendedor)
con.execute("""
CREATE OR REPLACE TABLE olist_fmt.sellers AS
SELECT
  CAST(seller_id            AS VARCHAR)       AS seller_id,
  CAST(total_orders         AS INTEGER)       AS seller_orders,
  CAST(total_items          AS INTEGER)       AS seller_items,
  CAST(total_gmv            AS DECIMAL(18,2)) AS seller_gmv,
  CAST(avg_delivery_days    AS DECIMAL(10,2)) AS seller_avg_delivery_days,
  CAST(avg_delay            AS DECIMAL(10,2)) AS seller_avg_delay,
  CAST(avg_review_score     AS DECIMAL(10,2)) AS seller_avg_review
FROM olist.vw_sellers;
""")

In [15]:
# CATEGORIES (categoría–mes)
con.execute("""
CREATE OR REPLACE TABLE olist_fmt.categories AS
SELECT
  CAST(product_category_name_english AS VARCHAR)   AS product_category_name_english,
  CAST(order_year                     AS INTEGER)   AS order_year,
  CAST(order_month                    AS VARCHAR)   AS order_month,
  CAST(category_items                 AS INTEGER)   AS category_items,
  CAST(category_orders                AS INTEGER)   AS category_orders,
  CAST(category_gmv                   AS DECIMAL(18,2)) AS category_gmv,
  CAST(category_freight               AS DECIMAL(18,2)) AS category_freight
FROM olist.vw_categories;
""")

## Verificación

In [16]:
def describe(table):
    print(f"\n=== {table} ===")
    print(con.execute(f"DESCRIBE {table}").fetchdf())
    print(con.execute(f"SELECT * FROM {table} LIMIT 5").fetchdf())

for t in [
    "olist_fmt.sales",
    "olist_fmt.logistics",
    "olist_fmt.customer_satisfaction",
    "olist_fmt.sellers",
    "olist_fmt.categories"
]:
    describe(t)


=== olist_fmt.sales ===
         column_name    column_type null   key default extra
0           order_id        VARCHAR  YES  None    None  None
1    items_per_order        INTEGER  YES  None    None  None
2              price  DECIMAL(18,2)  YES  None    None  None
3      freight_value  DECIMAL(18,2)  YES  None    None  None
4  order_total_value  DECIMAL(18,2)  YES  None    None  None
5       seller_count        INTEGER  YES  None    None  None
6         order_year        INTEGER  YES  None    None  None
7        order_month        VARCHAR  YES  None    None  None
8          order_dow        INTEGER  YES  None    None  None
                           order_id  items_per_order   price  freight_value  \
0  54282e97f61c23b78330c15b154c867d                1  145.00          21.46   
1  35a972d7f8436f405b56e36add1a7140                1   84.99           8.76   
2  03ef5dedbe7492bdae72eec50764c43f                1   24.90           8.33   
3  168626408cb32af0ffaf76711caae1dc              

## Guardado

In [17]:
import os, shutil

def qp(path):
    return os.path.abspath(path).replace("\\", "/")

out = "../../data/formatted/"
os.makedirs(out, exist_ok=True)

# -------- Parquet (único por tabla) --------
tables = ["sales","logistics","customer_satisfaction","sellers","categories"]
for name in tables:
    dst = qp(os.path.join(out, f"{name}.parquet"))
    con.execute(f"""
        COPY (SELECT * FROM olist_fmt.{name})
        TO '{dst}'
        (FORMAT PARQUET, COMPRESSION 'zstd');
    """)
    print(f"Parquet escrito: {dst}")


print("Formateo y exportación listos:", out)

Parquet escrito: c:/Users/Lizsa/OneDrive/Documents/GitHub/olist-brazil-ecommerce-analytics/data/formatted/sales.parquet
Parquet escrito: c:/Users/Lizsa/OneDrive/Documents/GitHub/olist-brazil-ecommerce-analytics/data/formatted/logistics.parquet
Parquet escrito: c:/Users/Lizsa/OneDrive/Documents/GitHub/olist-brazil-ecommerce-analytics/data/formatted/customer_satisfaction.parquet
Parquet escrito: c:/Users/Lizsa/OneDrive/Documents/GitHub/olist-brazil-ecommerce-analytics/data/formatted/sellers.parquet
Parquet escrito: c:/Users/Lizsa/OneDrive/Documents/GitHub/olist-brazil-ecommerce-analytics/data/formatted/categories.parquet
Formateo y exportación listos: ../../data/formatted/


In [18]:
con.close()